# NB_01 — Absorber Manufacturing Synthesis

This notebook uses the completed engineering source records and derives cross-source synthesis from their structured evidence:

```text
SOURCE_00_becker_transition_models.yaml
SOURCE_01_bismuth_microstructure.yaml
SOURCE_02_eliminating_nongaussian_spectral_response.yaml
```

It does **not** re-read the papers. It treats the source records as the engineering evidence layer and asks:

- Which variables recur across sources?
- Which relationships are supported by more than one source?
- Which quantitative values can be compared now?
- Which manufacturing specifications are supported?
- Which specifications remain unresolved?
- What should the next engineering notebook measure or model?

Outputs are written to:

```text
outputs/engineering_questions/absorber_manufacturing/SYNTHESIS_01/
```

and packaged as:

```text
exports/SYNTHESIS_01_export.zip
```

Run from top to bottom.


### v1.1 refinement

Candidate and open specifications are now generated from the loaded source records through transparent concept rules. They are no longer stored as hand-written source lists in the synthesis cells. The notebook preserves the distinction between source evidence and synthesis rules.


## 1. Configuration and repository paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]

SYNTHESIS_ID = "SYNTHESIS_01"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / "engineering_navigator").is_dir()
        ):
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(target)],
                check=True,
            )
        return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. "
        "Set REPO_ROOT_OVERRIDE to the repository path."
    )


REPO_ROOT = find_repo_root()
SOURCE_DIR = (
    REPO_ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / SYNTHESIS_ID
)
EXPORT_DIR = REPO_ROOT / "exports" / SYNTHESIS_ID
EXPORT_ZIP = REPO_ROOT / "exports" / f"{SYNTHESIS_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Sources   : {SOURCE_DIR.relative_to(REPO_ROOT)}")
print(f"Outputs   : {OUTPUT_DIR.relative_to(REPO_ROOT)}")


## 2. Load and validate the three source records

In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing source record: {path}")

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(
            f"{path.name}: expected one top-level mapping"
        )

    return data


records = {}

for filename in SOURCE_FILES:
    path = SOURCE_DIR / filename
    record = load_yaml(path)

    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(
            f"{filename}: missing source_id"
        )

    if source_id in records:
        raise ValueError(
            f"Duplicate source_id: {source_id}"
        )

    records[source_id] = record


if len(records) != len(SOURCE_FILES):
    raise ValueError(
        f"Expected {len(SOURCE_FILES)} source records, "
        f"loaded {len(records)}."
    )


status_rows = []

for source_id, record in records.items():
    status_rows.append(
        {
            "source_id": source_id,
            "title": record.get("title", ""),
            "record_status": record.get("record_status", ""),
            "extraction_status": record.get(
                "extraction_status",
                "",
            ),
            "reported_values": len(
                record.get("reported_values", [])
            ),
            "relationships": len(
                record.get(
                    "engineering_relationships",
                    [],
                )
            ),
        }
    )


status_df = (
    pd.DataFrame(status_rows)
    .sort_values("source_id")
    .reset_index(drop=True)
)

status_df

In [ ]:
incomplete = status_df[
    ~status_df["extraction_status"].astype(str).str.startswith("complete")
]

if not incomplete.empty:
    raise ValueError(
        "All three source records must be complete before synthesis:\n"
        + incomplete[["source_id", "extraction_status"]].to_string(index=False)
    )

print("Source-record validation: PASS")


## 3. Cross-source engineering variable matrix

The records use source-specific vocabulary. This section maps recurring engineering concepts onto a shared set of synthesis axes without replacing the original source terms.


In [ ]:
# Stable source slots come from SOURCE_FILES order, while canonical source_id
# values remain untouched in the YAML records.
source_slots = {}
for index, filename in enumerate(SOURCE_FILES):
    path = SOURCE_DIR / filename
    record = load_yaml(path)
    slot = f"SOURCE_{index:02d}"
    source_slots[slot] = record["source_id"]

slot_records = {
    slot: records[canonical_id]
    for slot, canonical_id in source_slots.items()
}

SYNTHESIS_AXES = {
    "critical_temperature": {
        "aliases": {"Tc"},
        "keywords": {"critical temperature"},
    },
    "absorber_thickness": {
        "aliases": {"Bi_thickness", "Au_thickness"},
        "keywords": {"thickness", "absorber thickness"},
    },
    "deposition_method": {
        "aliases": {"deposition_method"},
        "keywords": {"deposition", "electroplat", "evaporat"},
    },
    "grain_size": {
        "aliases": {
            "grain_size",
            "SEM_grain_size",
            "diffraction_grain_size",
            "average_grain_size",
            "average_grain_radius",
        },
        "keywords": {"grain size", "grain-size", "microstructure", "morphology"},
    },
    "heat_capacity": {
        "aliases": {"C"},
        "keywords": {"heat capacity"},
    },
    "thermal_conductance": {
        "aliases": {"G"},
        "keywords": {"thermal conductance"},
    },
    "spectral_response": {
        "aliases": {"delta_E", "tail_fraction", "low_energy_tail"},
        "keywords": {"spectral", "low-energy tail", "tail fraction", "energy resolution"},
    },
    "photon_energy": {
        "aliases": {"photon_energy", "secondary_electron_cloud_size"},
        "keywords": {"photon energy", "x-ray energy", "secondary-electron"},
    },
}

def searchable_record_text(record: dict) -> str:
    chunks = []
    for field in (
        "design_variables",
        "engineering_relationships",
        "engineering_constraints",
        "future_questions",
        "unreported_variables",
    ):
        chunks.append(str(record.get(field, "")))
    return " ".join(chunks).lower()

matrix_rows = []
for axis, rule in SYNTHESIS_AXES.items():
    row = {"engineering_axis": axis}
    support_count = 0

    for slot, record in slot_records.items():
        canonical_id = record["source_id"]
        variable_ids = {
            item.get("id")
            for item in record.get("design_variables", [])
            if isinstance(item, dict) and item.get("id")
        }
        alias_hits = sorted(variable_ids.intersection(rule["aliases"]))
        text = searchable_record_text(record)
        keyword_hits = sorted(
            keyword for keyword in rule["keywords"]
            if keyword.lower() in text
        )

        evidence = alias_hits or keyword_hits
        row[canonical_id] = ", ".join(alias_hits or keyword_hits) if evidence else "—"
        support_count += int(bool(evidence))

    row["source_count"] = support_count
    matrix_rows.append(row)

variable_matrix = (
    pd.DataFrame(matrix_rows)
    .sort_values(["source_count", "engineering_axis"], ascending=[False, True])
    .reset_index(drop=True)
)
variable_matrix


## 4. Quantitative evidence table

In [ ]:
value_rows = []

for source_id, record in records.items():
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue
        value_rows.append(
            {
                "source_id": source_id,
                "object": item.get("object"),
                "variable": item.get("variable"),
                "value": item.get("value"),
                "unit": item.get("unit"),
                "condition": item.get("condition"),
                "source_page": item.get("source_page"),
            }
        )

values_df = pd.DataFrame(value_rows)

FOCUS_VARIABLES = {
    "Tc",
    "Bi_thickness",
    "C",
    "G",
    "SEM_grain_size",
    "diffraction_grain_size",
    "average_grain_size",
    "average_grain_radius",
    "quantum_efficiency",
    "residual_resistance_ratio",
    "cloud_size",
    "delta_E",
    "predicted_delta_E",
}

focus_values = (
    values_df[values_df["variable"].isin(FOCUS_VARIABLES)]
    .sort_values(["variable", "source_id", "object"])
    .reset_index(drop=True)
)

focus_values


## 5. Derive recurring engineering relationships

Every source-level `engineering_relationship` is loaded directly from YAML. The notebook maps each relationship onto transparent engineering concepts using keyword rules, then reports where the same concept appears in multiple independent source records.

The concept rules are synthesis logic; the relationship text remains source-derived.


In [ ]:
RELATIONSHIP_CONCEPT_RULES = {
    "deposition_to_microstructure": {
        "required_any": [
            {"deposition", "electroplat", "evaporat"},
            {"grain", "microstructure", "morphology"},
        ],
    },
    "microstructure_to_trapping": {
        "required_any": [
            {"grain", "microstructure", "morphology"},
            {"trap", "scatter", "thermalization"},
        ],
    },
    "trapping_to_spectral_response": {
        "required_any": [
            {"trap", "thermalization"},
            {"tail", "spectral", "reconstructed event energy"},
        ],
    },
    "thickness_to_detector_response": {
        "required_any": [
            {"thickness"},
            {"tail", "quantum efficiency", "stopping power", "spectral"},
        ],
    },
    "thermal_design_coupling": {
        "required_any": [
            {"heat capacity", "thermal conductance", " c ", " g "},
            {"resolution", "pulse", "thermal", "geometry"},
        ],
    },
}

def relationship_text(item: dict) -> str:
    return (
        f"{item.get('relationship', '')} "
        f"{item.get('engineering_effect', '')}"
    ).lower()

def concept_matches(text: str, rule: dict) -> bool:
    # Each group requires at least one keyword hit.
    for group in rule["required_any"]:
        if not any(keyword in text for keyword in group):
            return False
    return True

relationship_rows = []
for source_id, record in records.items():
    for index, item in enumerate(record.get("engineering_relationships", [])):
        if not isinstance(item, dict):
            continue
        text = relationship_text(item)
        matched = [
            concept
            for concept, rule in RELATIONSHIP_CONCEPT_RULES.items()
            if concept_matches(text, rule)
        ]
        relationship_rows.append(
            {
                "source_id": source_id,
                "relationship_index": index,
                "relationship": item.get("relationship", ""),
                "engineering_effect": item.get("engineering_effect", ""),
                "source_pages": item.get("source_pages", []),
                "concepts": matched,
            }
        )

source_relationships_df = pd.DataFrame(relationship_rows)

concept_rows = []
for concept in RELATIONSHIP_CONCEPT_RULES:
    supporting = source_relationships_df[
        source_relationships_df["concepts"].apply(
            lambda concepts: concept in concepts
        )
    ]
    sources = sorted(supporting["source_id"].unique().tolist())
    concept_rows.append(
        {
            "concept": concept,
            "source_count": len(sources),
            "sources": sources,
            "relationship_count": len(supporting),
            "status": (
                "supported_across_sources"
                if len(sources) >= 2
                else "single_source_support"
            ),
        }
    )

relationships_df = (
    pd.DataFrame(concept_rows)
    .sort_values(["source_count", "concept"], ascending=[False, True])
    .reset_index(drop=True)
)

relationships_df


## 6. Generate candidate leading specifications

Candidate specifications are emitted only where the loaded source records satisfy an explicit evidence rule. The wording below is templated from the detected concept; the `evidence` field is computed from the contributing source records.


In [ ]:
SPEC_RULES = {
    "deposition_to_microstructure": {
        "min_sources": 2,
        "specification": (
            "Treat bismuth deposition method as a controlled absorber-manufacturing "
            "variable because it changes microstructure."
        ),
        "next_validation": (
            "Quantify process-to-process and batch-to-batch microstructure distributions."
        ),
    },
    "microstructure_to_trapping": {
        "min_sources": 2,
        "specification": (
            "Track absorber grain size and morphology as manufacturing acceptance variables."
        ),
        "next_validation": (
            "Determine a quantitative grain-size or morphology threshold tied to detector response."
        ),
    },
    "trapping_to_spectral_response": {
        "min_sources": 2,
        "specification": (
            "Use low-energy spectral tailing as a validation measurement for absorber thermalization."
        ),
        "next_validation": (
            "Measure spectral-tail fraction across replicated absorber batches."
        ),
    },
    "thickness_to_detector_response": {
        "min_sources": 2,
        "specification": (
            "Treat absorber thickness as a coupled stopping-power and thermalization variable."
        ),
        "next_validation": (
            "Run a controlled thickness series with efficiency and spectral-response measurements."
        ),
    },
    "thermal_design_coupling": {
        "min_sources": 2,
        "specification": (
            "Validate absorber-manufacturing changes with detector thermal variables rather than absorber geometry alone."
        ),
        "next_validation": (
            "Preserve matched TES and membrane conditions while measuring C, G, timing, and spectral response."
        ),
    },
}

concept_index = relationships_df.set_index("concept").to_dict("index")

candidate_specifications = []
spec_number = 1
for concept, rule in SPEC_RULES.items():
    result = concept_index.get(concept, {})
    if result.get("source_count", 0) < rule["min_sources"]:
        continue

    candidate_specifications.append(
        {
            "spec_id": f"SPEC_AM_{spec_number:02d}",
            "concept": concept,
            "specification": rule["specification"],
            "evidence": result["sources"],
            "source_count": result["source_count"],
            "state": "source_supported_candidate",
            "next_validation": rule["next_validation"],
        }
    )
    spec_number += 1

specifications_df = pd.DataFrame(candidate_specifications)
specifications_df


## 7. Derive unresolved specifications and next measurements

Open specifications are generated from two signals in the source records:

1. explicit `unreported_variables`; and
2. design variables that recur as engineering concepts but have no reported tolerance, range, distribution, repeatability, or yield criterion.

The normalization rules below group differently worded gaps into engineering categories.


In [ ]:
OPEN_SPEC_RULES = {
    "grain_size_acceptance": {
        "keywords": {"grain", "morphology"},
        "open_specification": "Absorber grain-size / morphology acceptance range",
        "next_measurement": "Measure grain-size distributions against spectral-tail fraction across replicated devices.",
    },
    "thickness_process_window": {
        "keywords": {"thickness"},
        "open_specification": "Electroplated-Bi thickness process window",
        "next_measurement": "Run a controlled thickness series measuring quantum efficiency, tailing, C, and energy resolution.",
    },
    "process_tolerances": {
        "keywords": {"tolerance", "current", "voltage", "plating", "bath", "rate"},
        "open_specification": "Electroplating process tolerances",
        "next_measurement": "Link process-parameter variation to grain size, morphology, yield, and spectral response.",
    },
    "repeatability": {
        "keywords": {"wafer-to-wafer", "batch-to-batch", "repeatability", "distribution"},
        "open_specification": "Manufacturing repeatability",
        "next_measurement": "Measure replicated wafer/batch distributions for key process and detector variables.",
    },
    "yield": {
        "keywords": {"yield", "failure", "rework"},
        "open_specification": "Manufacturing yield",
        "next_measurement": "Define pass/fail criteria and measure yield across process batches.",
    },
}

gap_rows = []
for source_id, record in records.items():
    for gap in record.get("unreported_variables", []):
        gap_rows.append(
            {
                "source_id": source_id,
                "gap": str(gap),
            }
        )

gaps_df = pd.DataFrame(gap_rows)

open_items = []
for concept, rule in OPEN_SPEC_RULES.items():
    if gaps_df.empty:
        continue

    mask = gaps_df["gap"].str.lower().apply(
        lambda text: any(keyword in text for keyword in rule["keywords"])
    )
    matches = gaps_df[mask]
    if matches.empty:
        continue

    supporting_sources = sorted(matches["source_id"].unique().tolist())
    open_items.append(
        {
            "concept": concept,
            "open_specification": rule["open_specification"],
            "why_open": "; ".join(sorted(matches["gap"].unique().tolist())),
            "gap_sources": supporting_sources,
            "source_count": len(supporting_sources),
            "next_measurement": rule["next_measurement"],
        }
    )

open_specs_df = (
    pd.DataFrame(open_items)
    .sort_values(["source_count", "open_specification"], ascending=[False, True])
    .reset_index(drop=True)
)

open_specs_df


## 8. Engineering synthesis

The source records now generate the engineering direction rather than merely supplying values to hand-written conclusions.

The synthesis asks whether the evidence currently supports this progression:

```text
Deposition process
        ↓
Bi thickness + grain size + morphology
        ↓
Carrier thermalization
        ↓
Spectral response + detector thermal response
        ↓
Acceptance criteria
        ↓
Repeatability + yield
```

The next notebook is selected from unresolved specifications rather than from a fixed source-reading sequence.


In [ ]:
NEXT_NOTEBOOK_RULES = [
    {
        "open_concept": "thickness_process_window",
        "id": "NB_02_ELECTROPLATED_BI_PROCESS_WINDOW",
        "engineering_question": (
            "What electroplated-Bi thickness and microstructure window preserves "
            "tail-free spectral response while increasing x-ray stopping power?"
        ),
        "outputs": [
            "candidate thickness range",
            "candidate grain-size acceptance metric",
            "required process measurements",
            "validation experiment design",
        ],
    },
    {
        "open_concept": "process_tolerances",
        "id": "NB_02_ELECTROPLATING_PROCESS_TOLERANCES",
        "engineering_question": (
            "Which electroplating process variables most strongly constrain absorber microstructure and detector response?"
        ),
        "outputs": [
            "candidate process variables",
            "measurement matrix",
            "process-window experiment design",
        ],
    },
]

open_concepts = set(open_specs_df["concept"]) if not open_specs_df.empty else set()

next_notebook = None
for rule in NEXT_NOTEBOOK_RULES:
    if rule["open_concept"] in open_concepts:
        next_notebook = {
            "id": rule["id"],
            "engineering_question": rule["engineering_question"],
            "inputs": [
                filename
                for filename in SOURCE_FILES
                if filename.startswith(("SOURCE_01", "SOURCE_02"))
            ],
            "outputs": rule["outputs"],
        }
        break

if next_notebook is None:
    next_notebook = {
        "id": "NB_02_ABSORBER_MANUFACTURING_REFINEMENT",
        "engineering_question": "Which unresolved absorber-manufacturing specification should be evaluated next?",
        "inputs": SOURCE_FILES,
        "outputs": ["ranked unresolved specifications", "next measurement plan"],
    }

next_notebook


## 9. Write synthesis outputs and export ZIP

In [ ]:
written_files = {}

status_csv = OUTPUT_DIR / "source_status.csv"
variable_matrix_csv = OUTPUT_DIR / "variable_matrix.csv"
focus_values_csv = OUTPUT_DIR / "quantitative_evidence.csv"
source_relationships_csv = OUTPUT_DIR / "source_relationships.csv"
relationships_csv = OUTPUT_DIR / "synthesis_relationships.csv"
specifications_csv = OUTPUT_DIR / "candidate_specifications.csv"
open_specs_csv = OUTPUT_DIR / "open_specifications.csv"
synthesis_json = OUTPUT_DIR / "synthesis_summary.json"

status_df.to_csv(status_csv, index=False)
variable_matrix.to_csv(variable_matrix_csv, index=False)
focus_values.to_csv(focus_values_csv, index=False)
source_relationships_df.to_csv(source_relationships_csv, index=False)
relationships_df.to_csv(relationships_csv, index=False)
specifications_df.to_csv(specifications_csv, index=False)
open_specs_df.to_csv(open_specs_csv, index=False)

synthesis_summary = {
    "synthesis_id": SYNTHESIS_ID,
    "sources": sorted(records),
    "source_slots": source_slots,
    "relationship_concepts": relationships_df.to_dict("records"),
    "candidate_specifications": candidate_specifications,
    "open_specifications": open_items,
    "next_notebook": next_notebook,
}
synthesis_json.write_text(
    json.dumps(synthesis_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

written_files = {
    "source_status": status_csv,
    "variable_matrix": variable_matrix_csv,
    "quantitative_evidence": focus_values_csv,
    "source_relationships": source_relationships_csv,
    "synthesis_relationships": relationships_csv,
    "candidate_specifications": specifications_csv,
    "open_specifications": open_specs_csv,
    "synthesis_summary": synthesis_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(REPO_ROOT)}")


In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 10. Handoff

If this synthesis runs successfully, inspect:

```text
candidate_specifications.csv
open_specifications.csv
source_relationships.csv
synthesis_relationships.csv
```

The candidate specifications should now change automatically as new source records add or remove cross-source support.

The `next_notebook` entry in `synthesis_summary.json` is selected from the unresolved engineering specifications detected in the evidence.

*Admissible generalizations trail leading specifications.*
